#### Preparation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision.models import convnext_small, ConvNeXt_Small_Weights
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import os

In [2]:
WEIGHTS_DIR = "../weights"

In [3]:
weights = ConvNeXt_Small_Weights.DEFAULT

train_transform = weights.transforms()
val_transform = weights.transforms()


In [4]:
train_dataset = datasets.ImageFolder(
    r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\val",
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

class_names = train_dataset.classes
num_classes = len(class_names)

print(class_names)


['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']


In [5]:
model = convnext_small(weights=weights)


In [6]:
in_features = model.classifier[2].in_features

model.classifier[2] = nn.Linear(in_features, num_classes)


#### Training 1 (Freeze-Backbone)

In [7]:
for param in model.features.parameters():
    param.requires_grad = False


In [8]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model.to(DEVICE)


ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [9]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=1e-3
)


In [10]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"\nTraining Epoch {epoch+1}/{epochs}", leave=False):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss, train_acc


In [11]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss, val_acc

In [12]:
EPOCHS_STAGE_FREEZE = 8

print("\n-----------Starting Phase 1 Training-----------\n")


for epoch in range(EPOCHS_STAGE_FREEZE):

    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_STAGE_FREEZE)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_STAGE_FREEZE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")



-----------Starting Phase 1 Training-----------




Training Epoch 1/8:  13%|█▎        | 4/30 [00:46<05:01, 11.59s/it]


KeyboardInterrupt: 

#### Training 2 (Fine-Tuning)

In [ ]:
for param in model.parameters():
    param.requires_grad = True


In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=1e-5)

In [ ]:
EPOCHS_STAGE_FINETUNE = 20

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(EPOCHS_STAGE_FINETUNE):

    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_STAGE_FINETUNE)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_STAGE_FINETUNE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state": model.state_dict(),
            "class_names": class_names
        }, os.path.join(WEIGHTS_DIR, "ConvNeXt-S.pth"))

#### Prediction Sample

In [ ]:
checkpoint = torch.load("convnext_cassava.pth")

class_names = checkpoint["class_names"]

model.load_state_dict(checkpoint["model_state"])
model.eval()

def predict(image_tensor):

    image_tensor = image_tensor.to(device)

    with torch.no_grad():
        outputs = model(image_tensor.unsqueeze(0))
        pred_idx = outputs.argmax(1).item()

    return class_names[pred_idx]
